# Proyecto: Algoritmo aplicado a Water Sort Puzzle

## Introducción

Nosotros seleccionamos el problema **Water Sort Game**, pues representa un desafío de optimización y combinatoria. Su estructura de juego basada en restricciones como colores, capacidades de los tubos y el cambio del liquido de tubo en tubo lo convierte en un problema muy interesante para evaluar cómo encontrar la secuencia óptima de movimientos sin caer en ciclos o explorar caminos redundantes (problemas de frontera).

Para resolverlo, vamos a implementar el algoritmo de búsqueda óptima **A* (A-Star)**. Elegimos este algoritmo porque, a diferencia de los métodos de búsqueda como BFS o DFS, A* combina el costo real acumulado con una función heurística informada ($f(n) = g(n) + h(n)$). Esto nos garantiza encontrar la solución óptima (la menor cantidad de movimientos para resolver el nivel) minimizando drásticamente el tiempo y el uso de memoria.

---

## Fases

El desarrollo se encuentra dividido en las siguientes cinco fases consecutivas:

* **Fase 1: Espacio de Estados**  
  Representar la estructura de los tubos, la distribución de los colores y las capacidades máximas de manera que Python y el algoritmo de búsqueda puedan interpretar y manipular cada configuración del juego de forma eficiente.

* **Fase 2: Reglas del Juego**  
  Definir restricciones legales para mover agua de un tubo a otro. A partir de estas reglas, se generan los estados siguientes válidos para continuar la exploración.

* **Fase 3: Diseño de la Heurística y Búsqueda**  
  Crear una función heurística $h(n)$ inteligente que estime la distancia hacia la solución. Esta función guiará al algoritmo A* para resolver el juego de manera rápida y óptima.

__Simultaneo al desarrollo del proyecto se hara uso de herramientas como canva para la creacion de la presentacion formal del proyecto.__


In [1]:
# Importar herramientas (bibliotecas)
import heapq
import itertools
import json

In [2]:
# 1. CLASES BASE

class Nodo:
    def __init__(self, estado, padre=None, accion=None, costo=0):
        self.estado = estado
        self.padre = padre
        self.accion = accion
        self.costo = costo

    def __str__(self):
        return f"Nodo(estado={self.estado}, accion={self.accion}, costo={self.costo})"


class Frontera:
    def __init__(self, f_eval):
        self.cola = []
        self.f_eval = f_eval
        self.contador = itertools.count()

    def add(self, node):
        heapq.heappush(self.cola, (self.f_eval(node), next(self.contador), node))

    def top(self):
        return self.cola[0][2]

    def pop(self):
        return heapq.heappop(self.cola)[2]

    def is_empty(self):
        return len(self.cola) == 0

def expand(problem, nodo):
    estado_act = nodo.estado
    nodos = [] 
    for (accion, next_state) in problem.result_actions(estado_act):
        cost = nodo.costo + problem.action_cost(estado_act, accion, next_state)
        nodos.append(Nodo(estado=next_state, padre=nodo, accion=accion, costo=cost))
    return nodos

## Modelado del Problema
Aquí definimos las reglas del universo del juego. Los estados serán tuplas inmutables representando los tubos y sus colores.

In [3]:
# 2. DEFINICIÓN DEL PROBLEMA: Water Sort

class WaterSortProblem:
    def __init__(self, initial_state, tube_capacity=4):
        """
        initial_state: Tupla de tuplas representando los tubos iniciales.
        tube_capacity: Capacidad máxima de cada tubo (por defecto 4).
        """
        self.initial = initial_state
        self.capacity = tube_capacity

    def is_goal(self, state):
        """
        Evalúa si el tablero actual es un estado ganador.
        Victoria = Todos los tubos están vacíos, o llenos con bloques del mismo color.
        """
        for tubo in state:
            if len(tubo) == 0:
                continue
            
            if len(tubo) == self.capacity and len(set(tubo)) == 1:
                continue
                
            return False
            
        return True

    def result_actions(self, state):
        """
        A partir de un estado, genera todos los movimientos legales.
        Retorna una lista de tuplas: [(accion, nuevo_estado), ...]
        La 'accion' se representa como una tupla (índice_tubo_origen, índice_tubo_destino).
        """
        sucesores = []
    
        for i, tubo_origen in enumerate(state):
            for j, tubo_destino in enumerate(state):
                
                if i == j:
                    continue
                if len(tubo_origen) == 0:
                    continue
                if len(tubo_destino) == self.capacity:
                    continue
                
                
                if len(tubo_destino) == 0 or tubo_origen[-1] == tubo_destino[-1]:
                    nuevo_origen = tubo_origen[:-1]
                    nuevo_destino = tubo_destino + (tubo_origen[-1],)
                    nuevo_estado_lista = list(state)
                    nuevo_estado_lista[i] = nuevo_origen
                    nuevo_estado_lista[j] = nuevo_destino
                    nuevo_estado_tupla = tuple(nuevo_estado_lista)
                    sucesores.append(((i, j), nuevo_estado_tupla))
                    
        return sucesores

    def action_cost(self, state, action, next_state):
        """
        El costo de hacer un movimiento.
        En Water Sort, cada movimiento cuesta exactamente lo mismo (1.0).
        """
        return 1.0

    def heuristic(self, state):
        """
        Función h(n): Estima cuántos movimientos faltan para ganar.
        Calcula las 'fronteras de color' (cambios de color dentro de un mismo tubo).
        Es admisible porque cada frontera obligará a hacer al menos un movimiento futuro.
        """
        costo_estimado = 0
        for tubo in state:
            for k in range(len(tubo) - 1):
                if tubo[k] != tubo[k + 1]:
                    costo_estimado += 1
        return float(costo_estimado)

## Algoritmo de Búsqueda: A*
Implementación del algoritmo de búsqueda del mejor primero configurado para funcionar como A*.

In [4]:
# 3. ALGORITMO A* Y RECONSTRUCCIÓN DE RUTA

def a_star_search(problem):
    """
    Ejecuta el algoritmo Best-First Search con la función f(n) = g(n) + h(n).
    """
    def f_eval(nodo):
        return nodo.costo + problem.heuristic(nodo.estado)
    FAIL = Nodo(estado="FAIL")
    node = Nodo(estado=problem.initial, padre=None, accion=None, costo=0)
    frontera = Frontera(f_eval)
    frontera.add(node)
    reached = {node.estado: node}
    while not frontera.is_empty():
        nodo_actual = frontera.pop()
        if problem.is_goal(nodo_actual.estado):
            exportar_grafo_json(reached)
            return nodo_actual
        expansion = expand(problem, nodo_actual)
        for hijo in expansion:
            est = hijo.estado
            if (est not in reached) or (hijo.costo < reached[est].costo):
                reached[est] = hijo
                frontera.add(hijo)        
    return FAIL

def path_actions(node):
    """
    A partir del nodo ganador, sube por los padres para extraer la secuencia de acciones.
    """
    if node.estado == "FAIL":
        return []
    acciones = []
    nodo_actual = node
    while nodo_actual is not None: 
        if nodo_actual.accion is not None:
            acciones.append(nodo_actual.accion)
        nodo_actual = nodo_actual.padre
        
    acciones.reverse()
    return acciones

In [7]:
def exportar_grafo_json(reached, archivo="grafo_zerethon.json"):
    """
    Toma la memoria del algoritmo A* y la convierte en un archivo JSON 
    estándar de Nodos y Conexiones para herramientas de visualización.
    """
    datos = {"nodes": [], "edges": []}
    
    for estado, nodo in reached.items():
        id_nodo = str(estado)
        datos["nodes"].append({"id": id_nodo, "label": id_nodo})
        
        if nodo.padre is not None:
            id_padre = str(nodo.padre.estado)
            texto_accion = f"Mover {nodo.accion[0]+1} a {nodo.accion[1]+1}"
            
            datos["edges"].append({
                "source": id_padre, 
                "target": id_nodo, 
                "label": texto_accion
            })
        
    with open(archivo, 'w') as f:
        json.dump(datos, f, indent=4)
    print(f"\n📂 ¡Archivo {archivo} generado con éxito para Zerethon Tools!")

## Interfaz Visual y Ejecución
Esta sección traduce el estado matemático de los tubos a un entorno visual utilizando emojis, y arranca la resolución del problema.

In [6]:
import time
import os

EMOJIS = {
    "Rojo": "🔴",
    "Azul": "🔵",
    "Verde": "🟢",
    "Amarillo": "🟡"
}

def imprimir_estado(estado, capacidad=4):
    """Imprime los tubos de ensayo en la consola de forma vertical."""
    print("\n")
    for nivel in range(capacidad - 1, -1, -1):
        fila = ""
        for tubo in estado:
            if len(tubo) > nivel:
                color = tubo[nivel]
                icono = EMOJIS.get(color, "⬛")
                fila += f" | {icono} | "
            else:
                fila += f" |    | "
        print(fila)
    
    base = "  ====  " * len(estado)
    print(base)
    
    numeros = "".join([f"   {i+1}    " for i in range(len(estado))])
    print(numeros)
    print("\n")

def animar_solucion(juego, estado_inicial, acciones):
    """
    Toma la lista de acciones ganadoras y las reproduce paso a paso, 
    limpiando la pantalla entre cada movimiento.
    """
    estado_actual = estado_inicial
    
    os.system('cls' if os.name == 'nt' else 'clear')
    
    print("ESTADO INICIAL:")
    imprimir_estado(estado_actual, juego.capacity)
    time.sleep(2)
    
    for paso, accion in enumerate(acciones, 1):
        origen, destino = accion
        
        lista_estado = list(estado_actual)
        color_movido = lista_estado[origen][-1]
        lista_estado[origen] = lista_estado[origen][:-1]
        lista_estado[destino] = lista_estado[destino] + (color_movido,)
        estado_actual = tuple(lista_estado)
        
        os.system('cls' if os.name == 'nt' else 'clear')
        print(f"PASO {paso}: Moviendo líquido del Tubo {origen + 1} al Tubo {destino + 1}...")
        imprimir_estado(estado_actual, juego.capacity)
        
        time.sleep(1.5) 
        
    print("<<Resuelto>>")

# 4. PRUEBA DEL ALGORITMO

nivel_inicial = (
    ("Rojo", "Azul", "Rojo", "Azul"),
    ("Azul", "Rojo", "Azul", "Rojo"),
    (),
    ()
)

juego = WaterSortProblem(initial_state=nivel_inicial, tube_capacity=4)

print("Calculando la ruta óptima...")
nodo_solucion = a_star_search(juego)

if nodo_solucion.estado != "FAIL":
    secuencia_ganadora = path_actions(nodo_solucion)
    animar_solucion(juego, nivel_inicial, secuencia_ganadora)
else:
    print("No se encontró solución para este nivel.")

Calculando la ruta óptima...

📂 ¡Archivo grafo_zerethon.json generado con éxito para Zerethon Tools!
ESTADO INICIAL:


 | 🔵 |  | 🔴 |  |    |  |    | 
 | 🔴 |  | 🔵 |  |    |  |    | 
 | 🔵 |  | 🔴 |  |    |  |    | 
 | 🔴 |  | 🔵 |  |    |  |    | 
  ====    ====    ====    ====  
   1       2       3       4    


PASO 1: Moviendo líquido del Tubo 1 al Tubo 3...


 |    |  | 🔴 |  |    |  |    | 
 | 🔴 |  | 🔵 |  |    |  |    | 
 | 🔵 |  | 🔴 |  |    |  |    | 
 | 🔴 |  | 🔵 |  | 🔵 |  |    | 
  ====    ====    ====    ====  
   1       2       3       4    


PASO 2: Moviendo líquido del Tubo 1 al Tubo 4...


 |    |  | 🔴 |  |    |  |    | 
 |    |  | 🔵 |  |    |  |    | 
 | 🔵 |  | 🔴 |  |    |  |    | 
 | 🔴 |  | 🔵 |  | 🔵 |  | 🔴 | 
  ====    ====    ====    ====  
   1       2       3       4    


PASO 3: Moviendo líquido del Tubo 1 al Tubo 3...


 |    |  | 🔴 |  |    |  |    | 
 |    |  | 🔵 |  |    |  |    | 
 |    |  | 🔴 |  | 🔵 |  |    | 
 | 🔴 |  | 🔵 |  | 🔵 |  | 🔴 | 
  ====    ====    ====    ==